In [2]:
# Load and inspect

import pandas as pd
import numpy as np

df = pd.read_csv('data/raw_mnch_data.csv')

print(df.shape)
print('------------------------------')
print(df.dtypes)
print('------------------------------')
print(df.isna().sum())
print('------------------------------')
print(df['Facility_Level'].value_counts())

(1140, 10)
------------------------------
Period                           object
Facility_ID                      object
County                           object
Facility_Level                   object
Target_Population_Pregnant        int64
ANC_1st_Visits                    int64
ANC_4th_Visits                  float64
Deliveries_Skilled_Attendant      int64
PNC_Visits_Within_48hrs         float64
Fully_Immunized_Children_FIC      int64
dtype: object
------------------------------
Period                           0
Facility_ID                      0
County                           0
Facility_Level                   0
Target_Population_Pregnant       0
ANC_1st_Visits                   0
ANC_4th_Visits                  56
Deliveries_Skilled_Attendant     0
PNC_Visits_Within_48hrs         38
Fully_Immunized_Children_FIC     0
dtype: int64
------------------------------
Facility_Level
Level 2    653
Level 3    284
Level 4    203
Name: count, dtype: int64


In [3]:
df.head()

,Period,Facility_ID,County,Facility_Level,Target_Population_Pregnant,ANC_1st_Visits,ANC_4th_Visits,Deliveries_Skilled_Attendant,PNC_Visits_Within_48hrs,Fully_Immunized_Children_FIC
0,2025-01-01,FAC_100,Machakos,Level 4,495,473,NaN,214,156.0,159
1,2025-01-01,FAC_101,Kilifi,Level 2,185,163,NaN,81,98.0,84
2,2025-01-01,FAC_102,Homa Bay,Level 2,207,208,102.0,100,120.0,101
3,2025-01-01,FAC_103,Kilifi,Level 2,238,248,268.0,157,179.0,154
4,2025-01-01,FAC_104,Kilifi,Level 3,340,276,198.0,182,137.0,115


In [4]:
# Standardize schema

df['Period'] = pd.to_datetime(df['Period'])

int_cols = [
    'Target_Population_Pregnant', 'ANC_1st_Visits', 'ANC_4th_Visits',
    'Deliveries_Skilled_Attendant', 'PNC_Visits_Within_48hrs',
    'Fully_Immunized_Children_FIC'
]
for col in int_cols:
    df[col] = df[col].astype('Int64')

print(df['Facility_ID'].nunique(), "unique facilities across", df['County'].nunique(), "counties")

100 unique facilities across 5 counties


In [ ]:
# Deduplicate and impute missing values

before = len(df)
df = df.drop_duplicates(subset=['Period', 'Facility_ID'])
print(f"Removed {before - len(df)} duplicate facility-month records")


df['ANC_4th_Visits'] = df['ANC_4th_Visits'].fillna(df.groupby(['County', 'Facility_Level'])['ANC_4th_Visits'].transform('median').round())
df['PNC_Visits_Within_48hrs'] = df['PNC_Visits_Within_48hrs'].fillna(df.groupby(['County', 'Facility_Level'])['PNC_Visits_Within_48hrs'].transform('median').round())

print(df[['ANC_4th_Visits', 'PNC_Visits_Within_48hrs']].isna().sum())

Removed 0 duplicate facility-month records
ANC_4th_Visits             0
PNC_Visits_Within_48hrs    0
dtype: int64


In [6]:
# Run logic validation checks and Data Quality flags

df['Error_ANC4_Exceeds_ANC1'] = df['ANC_4th_Visits'] > df['ANC_1st_Visits']
df['Warning_High_PNC_Ratio'] = df['PNC_Visits_Within_48hrs'] > (df['Deliveries_Skilled_Attendant'] * 1.20)

flag_count = df['Error_ANC4_Exceeds_ANC1'].astype(int) + df['Warning_High_PNC_Ratio'].astype(int)
df['Data_Quality_Pass'] = np.select([flag_count == 0, flag_count == 1, flag_count == 2], [100, 50, 0])

df['Facility_Any_Flag_This_Month'] = (flag_count > 0).astype(int)
chronic = df.groupby('Facility_ID')['Facility_Any_Flag_This_Month'].sum().rename('Facility_Flagged_Months_YTD')
df = df.merge(chronic, on='Facility_ID', how='left')

print("ANC4 > ANC1 error rate:", round(df['Error_ANC4_Exceeds_ANC1'].mean() * 100, 2), "%")
print("High PNC ratio warning rate:", round(df['Warning_High_PNC_Ratio'].mean() * 100, 2), "%")
print(df['Data_Quality_Pass'].value_counts(normalize=True).round(3) * 100)

ANC4 > ANC1 error rate: 5.96 %
High PNC ratio warning rate: 13.86 %
Data_Quality_Pass
100    81.4
50     17.4
0       1.2
Name: proportion, dtype: float64


In [13]:
# Calculate the 5 key indicators

df['ANC_Attrition_Rate_Pct'] = ((df['ANC_1st_Visits'] - df['ANC_4th_Visits']) / df['ANC_1st_Visits']) * 100
df['SBA_Coverage_Pct'] = (df['Deliveries_Skilled_Attendant'] / df['Target_Population_Pregnant']) * 100
df['PNC_Coverage_Pct'] = (df['PNC_Visits_Within_48hrs'] / df['Deliveries_Skilled_Attendant']) * 100
df['FIC_Coverage_Pct'] = (df['Fully_Immunized_Children_FIC'] / df['Target_Population_Pregnant']) * 100

dqi_overall = (df['Data_Quality_Pass'] == 100).mean() * 100
print(f"Overall DQI: {dqi_overall:.1f}%")

print(df[['ANC_Attrition_Rate_Pct', 'SBA_Coverage_Pct', 'PNC_Coverage_Pct', 'FIC_Coverage_Pct']].describe())

Overall DQI: 81.4%
       ANC_Attrition_Rate_Pct  SBA_Coverage_Pct  PNC_Coverage_Pct  \
count                  1140.0            1140.0            1140.0   
mean                39.271923         49.441454         99.125844   
std                 15.985376          13.11151         18.707875   
min                -17.813765         16.230366         51.794872   
25%                 31.988166         40.170898         84.127455   
50%                 40.940277         49.056034         98.688398   
75%                  50.49755         58.948817        113.490738   
max                 61.403509         85.779817        274.285714   

       FIC_Coverage_Pct  
count            1140.0  
mean          45.344446  
std           15.393611  
min           10.071942  
25%           34.062222  
50%           43.516976  
75%           55.019531  
max           96.610169  


In [10]:
# Save the cleaned, indicator-enriched dataset

df.to_csv('data/cleaned_mnch_data.csv', index=False)
print("Cleaned dataset saved:", df.shape)

Cleaned dataset saved: (1140, 19)
